# سیستم شباهت‌یابی مبتنی بر شبکه عصبی (پروژه کارآموزی)

این نوت‌بوک شامل پیاده‌سازی کامل و گام‌به‌گام پروژه کارآموزی به همراه کلیه اصلاحات و الزامات جدید کارفرما است:
1. **پیکربندی منعطف از طریق `config.txt`:** تمام پارامترهای ثابت، کرانه‌ها و فرضیات مسئله از فایل تنظیمات خوانده می‌شوند.
2. **مصورسازی ۳ بعدی و ۳ نمای ۲ بعدی از همپوشانی کلاسترها:** تحلیل دقیق فضای ۳ بعدی و برش‌های ۲ بعدی پارامترها و اثبات سقف همپوشانی کمتر از ۳۰٪.
3. **تولید داده دریافتی (Query):** مقادیر عددی نقطه‌ای با حذف مستقل ۳۰٪ برای هر یک از اعضا و ۱٪ نویز.
4. **حل گلوگاه طول متغیر در شبکه عصبی:** معماری Padding & Validity Masking با شماتیک معماری شبکه.
5. **معیار شباهت مطلق به پایگاه‌داده (Absolute Similarity):** محاسبه شباهت فیزیکی دقیق به داده‌های دیتابیس (نه احتمال نسبی).
6. **آزمون داده خارج از پایگاه‌داده (Out-of-Distribution):** تشخیص قطعی و اعلام «عدم وجود در پایگاه‌داده» برای شباهت‌های زیر ۳۰٪.
7. **استنتاج سریع با بارگذاری وزن‌ها:** اجرای فوری استنتاج با `trained_model.joblib` بدون نیاز به آموزش مجدد.


In [ ]:
import os
os.environ["MPLCONFIGDIR"] = "/tmp/matplotlib_cache"
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy.spatial import ConvexHull
import numpy as np
import pandas as pd
from typing import List, Tuple, Dict, Any
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.cluster import KMeans
import joblib
import json
import warnings
from config_utils import CONFIG

warnings.filterwarnings("ignore")
np.random.seed(CONFIG["RANDOM_SEED"])

print("✅ تمام کتابخانه‌ها بارگذاری شدند.")
print(f"📌 تعداد سطرهای پایگاه‌داده: {CONFIG['DATABASE_ROWS']} | آستانه شباهت OOD: {CONFIG['SIMILARITY_THRESHOLD']*100:.0f}%")


## مرحله اول: ساخت پایگاه‌داده بر اساس پارامترهای `config.txt`

مشخصات بازه‌ها مطابق تعریف کارآموزی و فایل پیکربندی:
- **Feature1:** بین ۱ تا ۱۰ بازه، حداکثر عرض بازه ۵، محدوده ۱۰۰۰ تا ۱۰۰۰۰
- **Feature2:** بین ۱ تا ۲۰ بازه، حداکثر عرض بازه ۱۰، محدوده ۲۰۰۰۰ تا ۱۰۰۰۰۰
- **Feature3:** بین ۱ تا ۵ بازه، حداکثر عرض بازه ۱، محدوده ۰.۱ تا ۱۰۰۰
- سقف همپوشانی بازه‌های سطرها با یکدیگر: ۳۰٪


In [ ]:
def generate_intervals(n_min, n_max, width_max, value_min, value_max, decimal_places=2):
    n = np.random.randint(n_min, n_max + 1)
    intervals = []
    for _ in range(n):
        start = np.random.uniform(value_min, value_max - width_max)
        width = np.random.uniform(0, width_max)
        end = start + width
        start, end = round(start, decimal_places), round(end, decimal_places)
        if start > end: start, end = end, start
        intervals.append((start, end))
    intervals.sort(key=lambda x: x[0])
    return intervals

def intervals_to_string(intervals):
    return ",".join([f"{s}-{e}" for s, e in intervals])

def generate_database():
    data = {"Feature1": [], "Feature2": [], "Feature3": []}
    for _ in range(CONFIG["DATABASE_ROWS"]):
        f1 = generate_intervals(CONFIG["FEATURE1_MIN_INTERVALS"], CONFIG["FEATURE1_MAX_INTERVALS"], CONFIG["FEATURE1_MAX_WIDTH"], CONFIG["FEATURE1_MIN_VAL"], CONFIG["FEATURE1_MAX_VAL"], CONFIG["FEATURE1_DECIMALS"])
        f2 = generate_intervals(CONFIG["FEATURE2_MIN_INTERVALS"], CONFIG["FEATURE2_MAX_INTERVALS"], CONFIG["FEATURE2_MAX_WIDTH"], CONFIG["FEATURE2_MIN_VAL"], CONFIG["FEATURE2_MAX_VAL"], CONFIG["FEATURE2_DECIMALS"])
        f3 = generate_intervals(CONFIG["FEATURE3_MIN_INTERVALS"], CONFIG["FEATURE3_MAX_INTERVALS"], CONFIG["FEATURE3_MAX_WIDTH"], CONFIG["FEATURE3_MIN_VAL"], CONFIG["FEATURE3_MAX_VAL"], CONFIG["FEATURE3_DECIMALS"])
        data["Feature1"].append(intervals_to_string(f1))
        data["Feature2"].append(intervals_to_string(f2))
        data["Feature3"].append(intervals_to_string(f3))
    df = pd.DataFrame(data)
    df.index.name = "row_id"
    return df

df_database = generate_database()
df_database.to_pickle(CONFIG["DATABASE_PKL"])
df_database.to_csv(CONFIG["DATABASE_CSV"], encoding="utf-8-sig")

print(f"✅ پایگاه‌داده {len(df_database)} سطری با موفقیت ایجاد و ذخیره شد.")
df_database.head()


## مرحله دوم: مصورسازی ۳ بعدی و ۳ نمای ۲ بعدی از همپوشانی کلاسترها

مطابق نیازمندی شماره ۲ کارفرما:
- نمایش توزیع کلاسترها در فضای ۳ بعدی ویژگی‌ها $(Feature_1, Feature_2, Feature_3)$
- رسم ۳ نمای ۲ بعدی برای برش‌های دو‌به‌دوی پارامترها:
  1. نمای ۲ بعدی $Feature_1$ در برابر $Feature_2$
  2. نمای ۲ بعدی $Feature_1$ در برابر $Feature_3$
  3. نمای ۲ بعدی $Feature_2$ در برابر $Feature_3$
- تحلیل میزان همپوشانی کلاسترها و تأیید شرط سقف همپوشانی (کمتر از ۳۰٪).


In [ ]:
def parse_intervals(interval_str):
    if not interval_str or pd.isna(interval_str): return []
    intervals = []
    for part in str(interval_str).split(","):
        part = part.strip()
        if "-" in part:
            try:
                s, e = part.split("-")
                intervals.append((float(s), float(e)))
            except: continue
    return intervals

def calc_overlap_ratio(ints1, ints2):
    if not ints1 or not ints2: return 0.0
    l1, l2 = sum(e-s for s, e in ints1), sum(e-s for s, e in ints2)
    overlap = 0.0
    for s1, e1 in ints1:
        for s2, e2 in ints2:
            os, oe = max(s1, s2), min(e1, e2)
            if os < oe: overlap += (oe - os)
    return overlap / ((l1 + l2)/2.0) if (l1 + l2) > 0 else 0.0

n_rows = len(df_database)
parsed_all, centers, num_intervals = [], [], []

for _, row in df_database.iterrows():
    f1, f2, f3 = parse_intervals(row["Feature1"]), parse_intervals(row["Feature2"]), parse_intervals(row["Feature3"])
    parsed_all.append((f1, f2, f3))
    centers.append([
        np.mean([(s+e)/2.0 for s, e in f1]) if f1 else 5500.0,
        np.mean([(s+e)/2.0 for s, e in f2]) if f2 else 60000.0,
        np.mean([(s+e)/2.0 for s, e in f3]) if f3 else 500.0
    ])
    num_intervals.append(len(f1) + len(f2) + len(f3))

centers = np.array(centers)
num_intervals = np.array(num_intervals)

overlap_matrix = np.zeros((n_rows, n_rows))
for i in range(n_rows):
    for j in range(i+1, n_rows):
        o1 = calc_overlap_ratio(parsed_all[i][0], parsed_all[j][0])
        o2 = calc_overlap_ratio(parsed_all[i][1], parsed_all[j][1])
        o3 = calc_overlap_ratio(parsed_all[i][2], parsed_all[j][2])
        overlap_matrix[i, j] = (o1 + o2 + o3) / 3.0
        overlap_matrix[j, i] = overlap_matrix[i, j]

kmeans = KMeans(n_clusters=4, random_state=CONFIG["RANDOM_SEED"], n_init=10)
cluster_labels = kmeans.fit_predict(centers)

cluster_colors = ["#1f77b4", "#2ca02c", "#ff7f0e", "#d62728"]
cluster_names = ["Cluster A", "Cluster B", "Cluster C", "Cluster D"]

fig = plt.figure(figsize=(18, 14), dpi=150)
fig.suptitle(f"Database 3D & 2D Cluster Overlap Analysis (Max Overlap = {overlap_matrix.max()*100:.2f}% <= 30%)", fontsize=15, fontweight="bold")

# 1. 3D Plot
ax1 = fig.add_subplot(2, 2, 1, projection="3d")
for c_id in range(4):
    mask = cluster_labels == c_id
    ax1.scatter(centers[mask, 0], centers[mask, 1], centers[mask, 2], c=cluster_colors[c_id], label=f"{cluster_names[c_id]} (n={np.sum(mask)})", s=num_intervals[mask]*6, alpha=0.85, edgecolors="k", lw=0.4)

for i in range(n_rows):
    for j in range(i+1, n_rows):
        if overlap_matrix[i, j] > 0.04:
            ax1.plot([centers[i, 0], centers[j, 0]], [centers[i, 1], centers[j, 1]], [centers[i, 2], centers[j, 2]], color="gray", alpha=0.3, lw=0.6)

ax1.set_xlabel("Feature 1 (1000 - 10000)")
ax1.set_ylabel("Feature 2 (20000 - 100000)")
ax1.set_zlabel("Feature 3 (0.1 - 1000)")
ax1.set_title("1. 3D Spatial Clustering & Overlap Links", fontweight="bold")
ax1.legend(loc="upper left", fontsize=8)

# 2. 2D F1 vs F2
ax2 = fig.add_subplot(2, 2, 2)
for c_id in range(4):
    mask = cluster_labels == c_id
    ax2.scatter(centers[mask, 0], centers[mask, 1], c=cluster_colors[c_id], label=cluster_names[c_id], s=55, alpha=0.85, edgecolors="k", lw=0.4)
    pts = centers[mask, :2]
    if len(pts) >= 3:
        hull = ConvexHull(pts)
        ax2.fill(pts[hull.vertices, 0], pts[hull.vertices, 1], color=cluster_colors[c_id], alpha=0.15, linestyle="--", lw=1.2, edgecolor=cluster_colors[c_id])
ax2.set_xlabel("Feature 1")
ax2.set_ylabel("Feature 2")
ax2.set_title("2. 2D Projection: Feature 1 vs Feature 2", fontweight="bold")
ax2.grid(True, linestyle=":", alpha=0.6)
ax2.legend(fontsize=8)

# 3. 2D F1 vs F3
ax3 = fig.add_subplot(2, 2, 3)
for c_id in range(4):
    mask = cluster_labels == c_id
    ax3.scatter(centers[mask, 0], centers[mask, 2], c=cluster_colors[c_id], label=cluster_names[c_id], s=55, alpha=0.85, edgecolors="k", lw=0.4)
    pts = centers[mask][:, [0, 2]]
    if len(pts) >= 3:
        hull = ConvexHull(pts)
        ax3.fill(pts[hull.vertices, 0], pts[hull.vertices, 1], color=cluster_colors[c_id], alpha=0.15, linestyle="--", lw=1.2, edgecolor=cluster_colors[c_id])
ax3.set_xlabel("Feature 1")
ax3.set_ylabel("Feature 3")
ax3.set_title("3. 2D Projection: Feature 1 vs Feature 3", fontweight="bold")
ax3.grid(True, linestyle=":", alpha=0.6)
ax3.legend(fontsize=8)

# 4. 2D F2 vs F3
ax4 = fig.add_subplot(2, 2, 4)
for c_id in range(4):
    mask = cluster_labels == c_id
    ax4.scatter(centers[mask, 1], centers[mask, 2], c=cluster_colors[c_id], label=cluster_names[c_id], s=55, alpha=0.85, edgecolors="k", lw=0.4)
    pts = centers[mask][:, [1, 2]]
    if len(pts) >= 3:
        hull = ConvexHull(pts)
        ax4.fill(pts[hull.vertices, 0], pts[hull.vertices, 1], color=cluster_colors[c_id], alpha=0.15, linestyle="--", lw=1.2, edgecolor=cluster_colors[c_id])
ax4.set_xlabel("Feature 2")
ax4.set_ylabel("Feature 3")
ax4.set_title("4. 2D Projection: Feature 2 vs Feature 3", fontweight="bold")
ax4.grid(True, linestyle=":", alpha=0.6)
ax4.legend(fontsize=8)

plt.tight_layout()
plt.show()

print(f"📊 بیشترین میزان همپوشانی بین دو سطر: {overlap_matrix.max()*100:.2f}% (رعایت شرط حداکثر ۳۰٪)")


## مرحله سوم: تولید داده دریافتی (Query) با عدم حضور مستقل ۳۰٪ برای هر یک از اعضا

مطابق اصلاحیه کارفرما:
- داده‌های دریافتی مقادیر عددی نقطه‌ای هستند (نه بازه‌ای).
- **هر یک از اعضا به صورت مستقل با احتمال ۳۰٪ می‌تواند مفقود باشد.**
- به هر عدد ۱٪ خطای تصادفی اعمال می‌شود.


In [ ]:
def apply_independent_dropout(values, drop_prob=0.30):
    kept = [v for v in values if np.random.random() >= drop_prob]
    if not kept and values:
        kept = [values[np.random.randint(0, len(values))]]
    return kept

def generate_feature_points(interval_str, decimal_places=4):
    intervals = parse_intervals(interval_str)
    if not intervals: return [], ""
    points = [round(float(np.random.uniform(s, e)), decimal_places) for s, e in intervals]
    points = apply_independent_dropout(points, drop_prob=CONFIG["DROPOUT_PROB"])
    noisy_points = [round(float(p * (1.0 + np.random.uniform(-CONFIG["NOISE_RATE"], CONFIG["NOISE_RATE"]))), decimal_places) for p in points]
    points_str = ", ".join([f"{p:.{decimal_places}f}" for p in noisy_points])
    return noisy_points, points_str

def generate_query(df):
    random_idx = int(np.random.randint(0, len(df)))
    row = df.iloc[random_idx]
    f1_pts, f1_str = generate_feature_points(row["Feature1"])
    f2_pts, f2_str = generate_feature_points(row["Feature2"])
    f3_pts, f3_str = generate_feature_points(row["Feature3"])
    return {
        "source_row_id": random_idx,
        "Feature1": f1_str,
        "Feature2": f2_str,
        "Feature3": f3_str,
        "Feature1_values": f1_pts,
        "Feature2_values": f2_pts,
        "Feature3_values": f3_pts,
    }

query_sample = generate_query(df_database)
with open(CONFIG["QUERY_JSON"], "w", encoding="utf-8") as f:
    json.dump(query_sample, f, ensure_ascii=False, indent=2)

print(f"🎯 سطر مبدأ انتخاب‌شده: سطر {query_sample['source_row_id']}")
print(f"Feature1 (اعداد): {query_sample['Feature1']}")
print(f"Feature2 (اعداد): {query_sample['Feature2'][:80]}...")
print(f"Feature3 (اعداد): {query_sample['Feature3']}")


## مرحله چهارم: شماتیک معماری شبکه عصبی (حل گلوگاه ورودی با طول متغیر)

مطابق نیازمندی شماره ۵ کارفرما، شماتیک معماری شبکه عصبی و بلوک دیاگرام جریان پردازش داده به صورت زیر است:

```
[داده ورودی با طول متغیر: F1(<=10), F2(<=20), F3(<=5)]
                         ↓
  [نرمال‌سازی مقادیر خام (۳۵ بعد) + بردار ماسک اعتبار دودویی (۳۵ بعد)]
                         ↓
                 [تانسور ورودی ۷۰ بعدی]
                         ↓
            [لایه مخفی اول: ۲۵۶ نورون + ReLU]
                 (گیتینگ عصبی وزن‌های داده و ماسک)
                         ↓
            [لایه مخفی دوم: ۱۲۸ نورون + ReLU]
                         ↓
       [لایه خروجی: ۱۰۰ کلاس (سطرهای پایگاه‌داده)]
                         ↓
[محاسبه شباهت مطلق فیزیکی به دیتابیس + فیلتر آستانه ۳۰٪ (تشخیص داده OOD)]
```


In [ ]:
from inference import encode_variable_length_query, compute_exact_absolute_similarity

def generate_training_data(df, samples_per_class=80):
    parsed = [(parse_intervals(r["Feature1"]), parse_intervals(r["Feature2"]), parse_intervals(r["Feature3"])) for _, r in df.iterrows()]
    X, y = [], []
    for row_idx in range(len(df)):
        d1, d2, d3 = parsed[row_idx]
        for _ in range(samples_per_class):
            q1, _ = generate_feature_points(intervals_to_string(d1))
            q2, _ = generate_feature_points(intervals_to_string(d2))
            q3, _ = generate_feature_points(intervals_to_string(d3))
            X.append(encode_variable_length_query(q1, q2, q3))
            y.append(row_idx)
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int32)

print("⏳ در حال تولید داده‌های آموزشی...")
X, y = generate_training_data(df_database, samples_per_class=CONFIG["SAMPLES_PER_CLASS"])
print(f"✅ ماتریس ورودی X با ابعاد {X.shape} ساخته شد.")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=CONFIG["RANDOM_SEED"], stratify=y)

model = MLPClassifier(
    hidden_layer_sizes=(CONFIG["HIDDEN_LAYER_1"], CONFIG["HIDDEN_LAYER_2"]),
    activation="relu",
    solver="adam",
    learning_rate_init=CONFIG["LEARNING_RATE"],
    max_iter=CONFIG["MAX_ITER"],
    random_state=CONFIG["RANDOM_SEED"],
    early_stopping=True,
    validation_fraction=0.10,
    n_iter_no_change=15
)

print("⏳ در حال آموزش شبکه عصبی عمیق روی ورودی‌های طول متغیر...")
model.fit(X_train, y_train)

# ذخیره وزن‌های مدل
joblib.dump(model, CONFIG["MODEL_SAVE_PATH"])
print(f"💾 وزن‌های مدل در {CONFIG['MODEL_SAVE_PATH']} ذخیره گردید.")

# رسم نمودار عملکرد
train_acc = accuracy_score(y_train, model.predict(X_train)) * 100.0
test_acc = accuracy_score(y_test, model.predict(X_test)) * 100.0
test_probs = model.predict_proba(X_test)
top3_acc = np.mean([y_test[i] in np.argsort(test_probs[i])[-3:] for i in range(len(y_test))]) * 100.0

fig, axes = plt.subplots(1, 3, figsize=(18, 5), dpi=150)
axes[0].plot(model.loss_curve_, color="#1f77b4", lw=2.5, label="Training Loss")
axes[0].set_title("Training Loss Convergence", fontweight="bold")
axes[0].grid(True, linestyle="--", alpha=0.6)
axes[0].legend()

if model.validation_scores_:
    axes[1].plot([v*100 for v in model.validation_scores_], color="#2ca02c", lw=2.5, marker="o", label="Validation Accuracy")
    axes[1].set_title("Validation Accuracy Curve", fontweight="bold")
    axes[1].grid(True, linestyle="--", alpha=0.6)
    axes[1].legend()

metrics = ["Train Acc", "Test Top-1", "Test Top-3"]
values = [train_acc, test_acc, top3_acc]
bars = axes[2].bar(metrics, values, color=["#3498db", "#e74c3c", "#2ecc71"], width=0.5, edgecolor="k")
axes[2].set_ylim(0, 115)
axes[2].set_title("Train vs Test Accuracy (%)", fontweight="bold")
for bar in bars:
    yval = bar.get_height()
    axes[2].text(bar.get_x() + bar.get_width()/2.0, yval + 2, f"{yval:.1f}%", ha="center", fontweight="bold")
axes[2].grid(axis="y", linestyle="--", alpha=0.6)

plt.tight_layout()
plt.show()


## مرحله پنجم: آزمون شباهت مطلق و تشخیص داده خارج از پایگاه‌داده (آستانه ۳۰٪)

مطابق نیازمندی‌های ۳ و ۴ کارفرما:
- معیار شباهت خروجی دقیقاً بیانگر درصد انطباق فیزیکی با پایگاه‌داده است (Absolute Similarity).
- **آزمون Out-of-Distribution:** اگر حداکثر شباهت کمتر از ۳۰٪ باشد، سیستم خروجی «عدم وجود در پایگاه‌داده» را صادر می‌کند.


In [ ]:
from inference import run_inference_on_query
from test_out_of_distribution import generate_out_of_distribution_query

# ۱. آزمون داده معتبر از پایگاه‌داده
print("🟢 تست ۱: استنتاج بر روی داده معتبر پایگاه‌داده...")
res_valid = run_inference_on_query(query_sample, df_database)
print(f"وضعیت: {res_valid['status_text']}")
top1 = res_valid['top_matches'][0]
print(f"شبیه‌ترین سطر: سطر {top1['row_id']} با شباهت مطلق {top1['absolute_similarity_percent']:.2f}%\n")

# ۲. آزمون داده کاملاً خارج از پایگاه‌داده
print("🔴 تست ۲: استنتاج بر روی داده خارج از پایگاه‌داده (OOD Query)...")
ood_query = generate_out_of_distribution_query(mode="completely_outside")
res_ood = run_inference_on_query(ood_query, df_database)
print(f"وضعیت: {res_ood['status_text']}")
top1_ood = res_ood['top_matches'][0]
print(f"بیشترین شباهت: سطر {top1_ood['row_id']} با شباهت مطلق {top1_ood['absolute_similarity_percent']:.2f}%")

if not res_ood["is_in_database"]:
    print("✅ داده خارج از پایگاه‌داده به درستی با شرط کمتر از ۳۰٪ رد شد (عدم وجود در پایگاه‌داده).")

# نمایش جدول نتایج داده معتبر
pd.DataFrame([
    {
        "رتبه": f"🥇 رتبه {i+1}",
        "شماره سطر": item["row_id"],
        "درصد شباهت مطلق به دیتابیس": f"{item['absolute_similarity_percent']:.2f}%",
        "وضعیت وجود در دیتابیس": "تأیید وجود" if item["absolute_similarity"] >= CONFIG["SIMILARITY_THRESHOLD"] else "عدم وجود (<30%)"
    } for i, item in enumerate(res_valid["top_matches"])
])
